<a href="https://colab.research.google.com/github/dacdemon/ITD/blob/main/Practica%202.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica 2: Procesamiento de Reportes de Planta

**Alumnos:** Damián Coria,

**Comisión:** 2

In [28]:
# Diagnóstico Inicial:

# Importamos pandas

import pandas as pd

# Cargamos el archivo csv desde un repositorio web

url = "https://raw.githubusercontent.com/dacdemon/ITD/refs/heads/main/practica_2.csv"
df = pd.read_csv(url)


In [29]:
# buscamos sus dimensiones totales y los tipos de datos
dimensiones = df.shape
tipos_de_datos = df.dtypes

print("Dimensiones del dataset:", dimensiones)
print("tipos de datos:")
print(tipos_de_datos)

Dimensiones del dataset: (4100, 9)
tipos de datos:
ID               int64
Datetime        object
Temperature    float64
Humidity       float64
Pressure       float64
Co2 Gas          int64
PM2.5          float64
PM10           float64
Daytime         object
dtype: object


In [30]:
# Tratamiento de valores duplicados
duplicados = df.duplicated().sum()
df.drop_duplicates(inplace=True)
duplicados_2 = df.duplicated().sum()
print("valores duplicados encontrados antes de la limpieza: ", duplicados)
print("Valores duplicados encontrados despues de la limpieza: ", duplicados_2)

valores duplicados encontrados antes de la limpieza:  0
Valores duplicados encontrados despues de la limpieza:  0


In [31]:
# Ajuste de fechas
df["Datetime"] = pd.to_datetime(df["Datetime"])
# Verificamos que se halla realizado el cambio
tiempo = df["Datetime"].dtype
print("El formato de la columna Datetime es: ", tiempo)



El formato de la columna Datetime es:  datetime64[ns]


In [44]:
# Análisis de tiempo de registros:
# A simple vista los datos parecen ordenados cronologicamente pero por su dimension no podemos estar seguros asique los ordenamos.

df = df.sort_values("Datetime").reset_index(drop=True)

# contestamos la primer pregunta: ¿Cuál es el período de tiempo total en el que hay registros?

fecha_inicio = df["Datetime"].min()
fecha_fin = df["Datetime"].max()

periodo_total = fecha_fin - fecha_inicio
print("PERÍODOS DE REGISTRO")
print("Primer registro: ", fecha_inicio)
print("Último registro: ", fecha_fin)
print("Período total: ", periodo_total)


PERÍODOS DE REGISTRO
Primer registro:  2019-05-20 19:08:34
Último registro:  2019-05-21 07:02:00
Período total:  0 days 11:53:26


In [49]:
# Calculamos la frecuencia de muestreo

df["Diferencia"] = df["Datetime"].diff()
print(df["Diferencia"].value_counts().head(10))

print("Promedios de frecuencia:")
print("Diferencia promedio entre registros:")
print(df["Diferencia"].mean())
print("Diferencia mediana entre registros:")
print(df["Diferencia"].median())

Diferencia
0 days 00:00:02    2012
0 days 00:00:01    1635
0 days 00:00:00     338
0 days 00:00:11      65
0 days 00:00:10      34
0 days 00:00:03       6
0 days 00:00:04       4
0 days 00:12:25       1
0 days 09:43:22       1
0 days 00:04:42       1
Name: count, dtype: int64
Promedios de frecuencia:
Diferencia promedio entre registros:
0 days 00:00:10.443034886
Diferencia mediana entre registros:
0 days 00:00:02


# ¿ Hay Gaps en los registros ?

La mediana calculada anteriormente representa mejor el intervalo de los datos.
Podemos considerar como Gap cualquier intervalo superior a 2 segundos.

In [67]:
# Buscaremos si existen diferencias mayores a 2 segundos:

def detectar_gaps(df):
    lista_gaps = []

    for diferencia in df["Diferencia"]:
        if diferencia > pd.Timedelta(seconds=2):
            lista_gaps.append(diferencia)
        else:
            pass

    return lista_gaps
lista_gaps = detectar_gaps(df)

conteo_gaps = len(lista_gaps)
print("Cantidad de gaps encontrados: ", conteo_gaps)


Cantidad de gaps encontrados:  114


## Variables sensadas
Temperature: Corresponde a la medición de la temperatura ambiental en °C (grados Celsius).

Humidity: Corresponde a la medición de la humedad en % (porcentaje de humedad relativa).

Pressure: Corresponde a la medición de la presión en psi .

Co2 Gas: Mide la concentración de dióxido de carbono en ppm (partes por millón).

PM2.5: Mide material particulado con un diámetro de hasta 2,5 µm.

PM10: Mide material particulado con un diámetro de hasta 10 µm.

Datetime: Permite conocer el momento exacto en que se realizó cada medición.

ID: Identifica de forma única cada registro.

## Conclusión

A partir del análisis realizado se pudo estudiar el comportamiento temporal de los datos registrados por los sensores.

En primer lugar, se realizó un diagnóstico inicial del dataset mediante `.shape` y `.dtypes`. Luego se verificaron y eliminaron los registros duplicados y se convirtió la columna `Datetime` desde texto al formato `datetime64`, permitiendo realizar operaciones relacionadas con el tiempo.

El período analizado comprende aproximadamente 11 horas y 53 minutos. El análisis de las diferencias entre registros permitió observar que la frecuencia de muestreo no es constante durante todo el dataset. Se identificaron períodos con registros cada 1 o 2 segundos, otros con intervalos de aproximadamente 10 segundos y también gaps importantes.

Desde el punto de vista de la calidad del dato, estos gaps son relevantes porque indican períodos sin registros y deben ser considerados antes de realizar análisis posteriores.

Finalmente, se identificaron como variables sensadas la temperatura, humedad, presión, concentración de CO2 y material particulado PM2.5 y PM10.

El procedimiento realizado se relaciona con el marco teórico de la Clase 2, donde se destaca la importancia de inspeccionar y limpiar los datos antes de utilizarlos para obtener resultados confiables.